# Member 3: evaluate, calibrate, export predicted evidence (Gate 2)
Needs a trained run folder on Drive (from the Member 2 notebook). All metrics are PIXEL level.
The test role (nepal-flooding) is scored ONCE at the end with the frozen model and temperature.
Exporting test-role evidence for the checker is not tuning and is fine before that.

In [ ]:
# Runtime > Change runtime type > T4 GPU first.
BRANCH = "m3/eval-export"   # top of the CV branch stack; switch to "main" once the PRs are merged
from google.colab import drive
drive.mount("/content/drive")
!rm -rf /content/FloodVerify
!git clone -q -b {BRANCH} https://github.com/aaarrvind/FloodVerify.git /content/FloodVerify
%cd /content/FloodVerify
!pip install -q -e . --no-deps jsonschema
import sys; sys.path.insert(0, "/content/FloodVerify/src")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Copy xBD to local disk (10-15 min). Skip if /content/xbd already exists in this session.
D = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
CK = f"{D}/checkpoints"          # run folders go here; never in git
X, MAN = "/content/xbd", "/content/xbd/rscc_flood_manifest.csv"
import os
if not os.path.exists(f"{X}/processed/evidence_labels"):
    !mkdir -p /content/xbd/processed
    !tar -xf "{D}/xbd_tars/train_images_labels_targets.tar" -C /content/xbd
    !tar -xf "{D}/xbd_tars/test_images_labels_targets.tar" -C /content/xbd
    !cp -r "{D}/xbd/hold" /content/xbd/
    !cp -r "{D}/xbd/tier3" /content/xbd/
    !cp "{D}/xbd/rscc_flood_manifest.csv" /content/xbd/
    !unzip -q "{D}/xbd/processed.zip" -d /content/xbd/processed
!ls /content/xbd/processed/evidence_labels | wc -l   # expect 10148

In [ ]:
RUN = "m1_siamese_unet_s0"
R = f"{CK}/{RUN}"
OUT = f"{D}/evidence_predicted/{RUN}"
!ls -la "{R}"

## Task 3 check first: ground-truth masks -> evidence vs Member 1's label evidence
Shows how much counting from a mask loses (touching buildings merge). This is the best case for predicted evidence.

In [ ]:
!python -m floodsitrep.cv.export_evidence --from-gt --xbd-root {X} --role val --out /content/ev_from_gt --schema evidence/schema/evidence.schema.json --label-dir {X}/processed/evidence_labels

## Tasks 1-2: validation metrics with 95% bootstrap CIs over tiles, fit temperature, ECE before/after

In [ ]:
!python -m floodsitrep.cv.eval.evaluate --checkpoint "{R}/best.pt" --xbd-root {X} --role val --fit-temperature

## Task 4 / Gate 2: predicted evidence for val and test -> Drive, schema-checked, then compare with label evidence

In [ ]:
for role in ("val", "test"):
    !python -m floodsitrep.cv.export_evidence --checkpoint "{R}/best.pt" --temperature "{R}/temperature.json" --xbd-root {X} --role {role} --out "{OUT}" --schema evidence/schema/evidence.schema.json
!ls "{OUT}" | wc -l
!python -c "from floodsitrep.cv.export_evidence import compare_with_labels as c; import json; print(json.dumps(c('{OUT}', '{X}/processed/evidence_labels'), indent=2))"

## Task 5 (END ONLY): held-out Nepal test metrics, frozen model

In [ ]:
# !python -m floodsitrep.cv.eval.evaluate --checkpoint "{R}/best.pt" --xbd-root {X} --role test